# 🖥️ Diffusion

In Eulerian ocean models, sub-grid scale dispersion of tracers such as heat, salt, or nutrients is often parameterized as a diffusive process. In Lagrangian particle simulations, sub-grid scale effects can be parameterized as a stochastic process, randomly displacing a particle position in proportion to the local eddy diffusivity ([Van Sebille et al. 2018](https://doi.org/10.1016/j.ocemod.2017.11.008)). Parameterizing sub-grid scale dispersion may be especially important when coarse velocity fields are used that do not resolve mesoscale eddies ([Shah et al., 2017](https://doi.org/10.1175/JPO-D-16-0098.1)). This tutorial explains how to use a sub-grid scale parameterization in _Parcels_ that is consistent with the advection-diffusion equation used in Eulerian models.

## Stochastic differential equations (SDE) consistent with advection-diffusion

The time-evolution of a stochastic process is described by a stochastic differential equation. The time-evolution of the conditional probability density of a stochastic process is described by a Fokker-Planck equation (FPE). The advection-diffusion equation, describing the evolution of a tracer, can be written as a Fokker-Planck equation. Therefore, we can formulate a stochastic differential equation for a particle in the Lagrangian frame undergoing advection with stochastic noise proportional to the local diffusivity in a way that is consistent with advection-diffusion in the Eulerian frame. For details, see [Shah et al., 2011](https://doi.org/10.1016/j.ocemod.2011.05.008) and [van Sebille et al., 2018](https://doi.org/10.1016/j.ocemod.2017.11.008).

The stochastic differential equation for a particle trajectory including diffusion is

$$
\begin{aligned}
    d\mathbf{X}(t) &\overset{\text{Îto}}{=} (\mathbf{u} + \nabla \cdot \mathbf{K}) dt + \mathbf{V}(t, \mathbf{X})\cdot d\mathbf{W}(t), \\
    \mathbf{X}(t_0) &= \mathbf{x}_0,
\end{aligned}
$$

where $\mathbf{X}$ is the particle position vector ($\mathbf{x}_0$ being the initial position vector), and $\mathbf{u}$ the velocity vector. $\mathbf{V}$ is calculated from the diffusivity tensor $\mathbf{K} = \frac{1}{2} \mathbf{V} \cdot \mathbf{V}^T$ and $d\mathbf{W}(t)$ a Wiener increment (normally distributed with zero mean and variance $dt$). Particle distributions obtained by solving the above equation are therefore consistent with Eulerian concentrations found by solving the advection-diffusion equation.

In three-dimensional ocean models diffusion operates along slopes of neutral buoyancy. To account for these slopes, the 3D diffusivity tensor $\mathbf{K}$ (and therefore $\mathbf{V}$) contains off-diagonal components. Three-dimensional advection-diffusion is not yet implemented in _Parcels_, but it is currently under development. Here we instead focus on the simpler case of diffusion in a horizontal plane, where diffusivity is specified only in the zonal and meridional direction, i.e.

$$
\mathbf{K}(x,y)=\begin{bmatrix}
K_x(x,y) & 0\\
0 & K_y(x,y)
\end{bmatrix}.
$$

The above stochastic differential equation then becomes

$$
\begin{align}
    dX(t) &= a_x dt + b_x dW_x(t), \quad &X(t_0) = x_0,\\
    dY(t) &= a_y dt + b_y dW_y(t), \quad &Y(t_0) = y_0,
\end{align}
$$

where $a_i = v_i + \partial_i K_i(x, y)$ is the deterministic drift term and $b_i = \sqrt{2K_i(x, y)}$ a stochastic noise term ($\partial_i$ denotes the partial derivative with respect to $i$).

## Numerical Approximations of SDEs

The simplest numerical approximation of the above SDEs is obtained by replacing $dt$ by a finite time discrete step $\Delta t$ and $dW$ by a discrete increment $\Delta W$, yielding the **Euler-Maruyama (EM) scheme** ([Maruyama, 1955](https://link.springer.com/article/10.1007/BF02846028)):

$$
\begin{equation}
    X_{n+1} = X_n + a_x \Delta t + b_x \Delta W_{n, x},
\end{equation}
$$

with a similar expression for $Y$.

A higher-order scheme is found by including extra terms from a Taylor expansion on our SDE, yielding the **Milstein scheme of order 1 (M1)**:

$$
\begin{equation}
    X_{n+1} = X_n + a_x \Delta t + b_x \Delta W_x + \frac{1}{2}b_x \partial_x b_x(\Delta W_{n, x}^2 - \Delta t),
\end{equation}
$$

which can be rewritten by explicitly writing $b_x\partial_x b_x$ as $\partial_x K_x(z)$:

$$
\begin{equation}
    X_{n+1} = X_n + v_x \Delta t + \frac{1}{2}\partial_x K_x(\Delta W_{n, x}^2 + \Delta t) + b\Delta W_n.
\end{equation}
$$

The extra term in the M1 scheme provides extra accuracy at negligible computational cost.

The spatial derivatives in the EM and M1 schemes can be approximated by a central difference. Higher order numerical schemes (see [Gräwe et al., 2012](https://doi.org/10.1007/s10236-012-0523-y)) include higher order derivatives. It should be possible to also use cubic spline interpolation in Parcels to compute the derivatives, which may be more accurate than finite differences. This is to be explored.

An overview of numerical approximations for SDEs in a particle tracking setting can be found in [Gräwe (2011)](https://doi.org/10.1016/j.ocemod.2010.10.002).

## Using Advection-Diffusion Kernels in Parcels

The EM and M1 advection-diffusion approximations are available as {py:func}`parcels.kernels.AdvectionDiffusionEM` and {py:func}`parcels.kernels.AdvectionDiffusionM1`, respectively. The {py:func}`parcels.kernels.AdvectionDiffusionM1` kernel should be the default choice, as the increased accuracy comes at negligible computational cost.

The advection component of these kernels is similar to that of the Explicit Euler advection kernel (`AdvectionEE`). In the special case where diffusivity is constant over the entire domain, the diffusion-only kernel {py:func}`parcels.kernels.DiffusionUniformKh` can be used in combination with an advection kernel of choice. Since the diffusivity here is space-independent, gradients are not calculated, increasing efficiency. The diffusion-step can in this case be computed after or before advection, thus allowing you to chain kernels in a list.

Just like velocities, diffusivities are passed to Parcels in the form of {py:func}`parcels.Field` objects. When using {py:func}`parcels.kernels.DiffusionUniformKh`, they should be added to the {py:func}`parcels.FieldSet` object as constant fields, e.g. `fieldset.add_constant_field("Kh_zonal", 1)`.

To make a central difference approximation for computing the gradient in diffusivity, a resolution for this approximation `dres` is needed: _Parcels_ approximates the gradients in diffusivities by using their values at the particle's location ± `dres` (in both $x$ and $y$). A value of `dres` must be specified and added to the FieldSet by the user (e.g. `fieldset.add_context("dres", 0.01)`). Currently, it is unclear what the best value of `dres` is. From experience, the size of `dres` should be smaller than the spatial resolution of the data, but within reasonable limits of machine precision to avoid numerical errors. We are working on a method to compute gradients differently so that specifying `dres` is not necessary anymore.

## Example: Impermeable Diffusivity Profile

Let's see the {py:func}`parcels.kernels.AdvectionDiffusionM1` in action and see why it's preferable over the {py:func}`parcels.kernels.AdvectionDiffusionEM` kernel. 

To do so, we create an idealized profile with diffusivities $K_\text{zonal}$ uniform everywhere ($K_\text{zonal} = \bar{K}=0.5$) and $K_\text{meridional}$ constant in the zonal direction, while having the following profile in the meridional direction:

$$
K_\text{meridional}(y) = \bar{K}\frac{2(1+\alpha)(1+2\alpha)}{\alpha^2H^{1+1/\alpha}} \begin{cases}
y(L-2y)^{1/\alpha},\quad 0 \leq y \leq L/2,\\
(L-y)(2y-1)^{1/a},\quad H/2 \leq y \leq L,
\end{cases}
$$

with $L$ being the basin length scale, $\alpha$ as a parameter determining the steepness in the gradient in the profile. This profile is similar to that used by [Gräwe (2011)](https://doi.org/10.1016/j.ocemod.2010.10.002), now used in the meridional direction for illustrative purposes.

Let's plot $K_\text{meridional}(y)$:


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import parcels
import parcels.tutorial

In [ ]:
K_bar = 0.5  # Average diffusivity
alpha = 1.0  # Profile steepness
L = 1.0  # Basin scale

# Number of grid cells in y_direction (101 +2)
# one level above and one below, where fields are set to zero
Ny = 103

# Spatial resolution
dy = 1.03 / Ny

# y-coordinates for grid
y = np.linspace(-0.01, 1.01, 103)

# y-coordinates used for setting diffusivity
y_K = np.linspace(0.0, 1.0, 101)

# Placeholder for fraction term in K(y) formula
beta = np.zeros(y_K.shape)

for yi in range(len(y_K)):
    if y_K[yi] < L / 2:
        beta[yi] = y_K[yi] * np.power(L - 2 * y_K[yi], 1 / alpha)
    elif y_K[yi] >= L / 2:
        beta[yi] = (L - y_K[yi]) * np.power(2 * y_K[yi] - L, 1 / alpha)
Kh_meridional = (
    0.1
    * (2 * (1 + alpha) * (1 + 2 * alpha))
    / (alpha**2 * np.power(L, 1 + 1 / alpha))
    * beta
)
Kh_meridional = np.concatenate((np.array([0]), Kh_meridional, np.array([0])))

In [ ]:
plt.plot(Kh_meridional, y)
plt.ylabel("y")
plt.xlabel(r"$K_{meridional}$")
plt.show()

In this profile, diffusivity drops to 0 at $y=0.5$ and at $y=0$ and $y=1$. In the absence of advection, particles starting out in one half of the domain should remain confined to that half as they are unable to cross the points where the diffusivity drops to 0. The line $y=0.5$ should therefore provide an impermeable barrier.

Now we can put this idealized profile into a flat fieldset:


In [ ]:
from parcels._datasets.structured.generated import simple_UV_dataset

ds = simple_UV_dataset(n_times=1, nz=1, ny=Ny, nx=1, mesh="flat")
ds["lat"][:] = np.linspace(-0.01, 1.01, Ny)
ds["Kh_meridional"] = (["YG", "XG"], Kh_meridional[:, None])
ds

In [ ]:
fieldset = parcels.FieldSet.from_sgrid_conventions(ds, mesh="flat")
fieldset.add_constant_field("Kh_zonal", 1)
fieldset.add_context("dres", 0.00005)

We release 100 particles at ($x=0$, $y=0.75$).


In [ ]:
def get_test_particles():
    return parcels.ParticleSet(
        fieldset,
        pclass=parcels.Particle,
        x=np.zeros(100),
        y=np.ones(100) * 0.75,
        t=np.repeat(np.timedelta64(0, "s"), 100),
    )

Now we will simulate the advection and diffusion of the particles using the {py:func}`parcels.kernels.AdvectionDiffusionM1` kernel. We run the simulation for 0.3 seconds, with a numerical timestep $\Delta t = 0.001$ s. We also write away particle locations at each timestep for plotting. Note that this will hinder a runtime comparison between kernels, since it will cause most time to be spent on I/O.


In [ ]:
testParticles = get_test_particles()
output_file = parcels.ParticleFile("M1_out.parquet", outputdt=np.timedelta64(1, "ms"))

testParticles.execute(
    parcels.kernels.AdvectionDiffusionM1,
    runtime=np.timedelta64(300, "ms"),
    dt=np.timedelta64(1, "ms"),
    output_file=output_file,
)

In [ ]:
M1_out = parcels.read_particlefile("M1_out.parquet")

We can plot the individual coordinates of the particle trajectories against time ($x$ against $t$ and $y$ against $t$) to investigate how diffusion works along each axis.


In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_figwidth(12)

x = np.arange(0, 0.3, 0.001)
for traj in M1_out.partition_by("particle_id", maintain_order=True):
    ax[0].plot(x, traj["y"][: len(x)], alpha=0.3)
ax[0].scatter(0, 0.75, s=20, c="r", zorder=3)
ax[0].set_ylabel("y")
ax[0].set_ylim(0, 1)

for traj in M1_out.partition_by("particle_id", maintain_order=True):
    ax[1].plot(x, traj["x"][: len(x)], alpha=0.3)
ax[1].scatter(0, 0, s=20, c="r", zorder=3)
ax[1].set_ylabel("x")
ax[1].set_ylim(-1, 1)

for ai in ax:
    ai.set_xlabel("t")
    ai.set_xlim(0, 0.3)

fig.suptitle(
    "`AdvectionDiffusionM1` Simulation: Particle trajectories in the x- and y-directions against time"
)
plt.show()

We see that the along the meridional direction, particles remain confined to the ‘upper’ part of the domain, not crossing the impermeable barrier where the diffusivity drops to zero. In the zonal direction, particles follow random walks, since all terms involving gradients of the diffusivity are zero.

Now let's execute the simulation with the {py:func}`parcels.kernels.AdvectionDiffusionEM` kernel instead.


In [ ]:
testParticles = get_test_particles()
output_file = parcels.ParticleFile("EM_out.parquet", outputdt=np.timedelta64(1, "ms"))
np.random.seed(1636)  # Random seed for reproducibility
testParticles.execute(
    parcels.kernels.AdvectionDiffusionEM,
    runtime=np.timedelta64(300, "ms"),
    dt=np.timedelta64(1, "ms"),
    output_file=output_file,
)

In [ ]:
EM_out = parcels.read_particlefile("EM_out.parquet")

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_figwidth(12)

x = np.arange(0, 0.3, 0.001)
x = np.arange(0, 0.3, 0.001)
for traj in EM_out.partition_by("particle_id", maintain_order=True):
    ax[0].plot(x, traj["y"][: len(x)], alpha=0.3)
ax[0].scatter(0, 0.75, s=20, c="r", zorder=3)
ax[0].set_ylabel("y")
ax[0].set_ylim(0, 1)

for traj in EM_out.partition_by("particle_id", maintain_order=True):
    ax[1].plot(x, traj["x"][: len(x)], alpha=0.3)
ax[1].scatter(0, 0, s=20, c="r", zorder=3)
ax[1].set_ylabel("x")
ax[1].set_ylim(-1, 1)

for ai in ax:
    ai.set_xlabel("t")
    ai.set_xlim(0, 0.3)

fig.suptitle(
    "`AdvectionDiffusionEM` Simulation: Particle trajectories in the x- and y-directions against time"
)
plt.show()

The Wiener increments for both simulations are equal, as they are fixed through a random seed. As we can see, the Euler-Maruyama scheme performs worse than the Milstein scheme, letting particles cross the impermeable barrier at $y=0.5$. In contrast, along the zonal direction, particles follow the same random walk as in the Milstein scheme, which is expected since the extra terms in the Milstein scheme are zero in this case.


## Using horizontal diffusion calculated from velocity fields

In the case when velocity fields are available, there is the possibility to calculate coefficients of diffusion based on closure parameterizations. The Smagorinsky method ([Smagorinsky, 1963](<https://doi.org/10.1175/1520-0493(1963)091%3C0099:GCEWTP%3E2.3.CO;2>)), which was originally proposed as a parameterization for horizontal eddy viscosity, is often used to parameterize horizontal eddy diffusivity as well. It computes the eddy diffusivity as

$$
K = C_s \Delta x \Delta y \sqrt{\left(\frac{\partial u}{\partial x}\right)^2 + \left(\frac{\partial v}{\partial y}\right)^2 + \frac{1}{2}\left(\frac{\partial u}{\partial y} +\frac{\partial v}{\partial x}\right)^2},
$$

where $C_s$, the Smagorinsky constant, is a dimensionless tuning parameter. It uses the grid area $\Delta x \Delta y$ as its spatial scale, and the norm of the strain rate tensor as its time scale, given as the square-rooted term.

Let’s see the example of implementation of the Smagorinsky method to the GlobalCurrents files of the region around South Africa. For simplicity, we are not taking gradients in the Smagorinsky-computed diffusivity field into account here.

First, create a new kernel for Smagorinsky diffusion method:


In [ ]:
def smagdiff(particles, fieldset):
    dx = 0.01
    # gradients are computed by using a local central difference.
    updx, vpdx = fieldset.UV[
        particles.t, particles.z, particles.y, particles.x + dx, particles
    ]
    umdx, vmdx = fieldset.UV[
        particles.t, particles.z, particles.y, particles.x - dx, particles
    ]
    updy, vpdy = fieldset.UV[
        particles.t, particles.z, particles.y + dx, particles.x, particles
    ]
    umdy, vmdy = fieldset.UV[
        particles.t, particles.z, particles.y - dx, particles.x, particles
    ]

    dudx = (updx - umdx) / (2 * dx)
    dudy = (updy - umdy) / (2 * dx)

    dvdx = (vpdx - vmdx) / (2 * dx)
    dvdy = (vpdy - vmdy) / (2 * dx)

    A = fieldset.cell_areas[particles.t, particles.z, particles.y, particles.x]
    sq_deg_to_sq_m = (1852 * 60) ** 2 * np.cos(particles.y * np.pi / 180)
    A = A / sq_deg_to_sq_m
    Kh = fieldset.Cs * A * np.sqrt(dudx**2 + 0.5 * (dudy + dvdx) ** 2 + dvdy**2)

    dy = np.random.normal(0.0, 1.0, particles.dt.shape) * np.sqrt(
        2 * np.fabs(particles.dt) * Kh
    )
    dx = np.random.normal(0.0, 1.0, particles.dt.shape) * np.sqrt(
        2 * np.fabs(particles.dt) * Kh
    )
    particles.dy += dy
    particles.dx += dx

Reading velocity fields from a Copernicus Marine Service dataset:


In [ ]:
# Load the CopernicusMarine data in the Agulhas region from the example_datasets
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
).isel(depth=slice(0, 1))

Adding parameters (`cell_areas` – areas of computational cells, and `Cs` – Smagorinsky constant) to `fieldset` that are needed for the `smagdiff` kernel


In [ ]:
def degree_lat_to_meter(d):
    return d * 1000.0 * 1.852 * 60.0


def degree_lon_to_meter(d, lat):
    return d * 1000.0 * 1.852 * 60.0 * np.cos(lat * np.pi / 180)


def calc_cell_areas(ds):
    """calculate cell areas for rectilinear grids"""
    lon, lat = ds["longitude"], ds["latitude"]
    assert "degrees" in lon.attrs["units"]
    assert "degrees" in lat.attrs["units"]

    LON, LAT = np.meshgrid(lon, lat)
    X, Y = degree_lon_to_meter(LON, LAT), degree_lat_to_meter(LAT)

    dX = np.gradient(X, axis=1)
    dY = np.gradient(Y, axis=0)
    cell_areas = dX * dY

    return cell_areas


fields = {"U": ds_fields["uo"], "V": ds_fields["vo"]}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)

ds_fset["cell_areas"] = (["lat", "lon"], calc_cell_areas(ds_fields))

fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)
fieldset.add_context("Cs", 0.1)

# Convert the FieldSet to windowed arrays for better performance
fieldset = fieldset.to_windowed_arrays()

In the example, particles are released at one location and simulated for 2 days using advection (`AdvectionRK2`) and diffusion (`smagdiff`) kernels.

In [ ]:
time = np.repeat(fieldset.time_interval.left, 13)
lon = [32] * len(time)
lat = [-30.1] * len(time)
z = [0] * len(time)
pset = parcels.ParticleSet(
    fieldset=fieldset,
    pclass=parcels.Particle,
    t=time,
    z=z,
    y=lat,
    x=lon,
)

output_file = parcels.ParticleFile("smagdiff.parquet", outputdt=np.timedelta64(1, "h"))

np.random.seed(1636)  # Random seed for reproducibility

pset.execute(
    [parcels.kernels.AdvectionRK2, smagdiff],
    runtime=np.timedelta64(2, "D"),
    dt=np.timedelta64(5, "m"),
    output_file=output_file,
)

When we visualize the trajectories, we can see that the particles released at the same time and location spread out over time:


In [ ]:
df = parcels.read_particlefile("smagdiff.parquet")

temperature = ds_fields.isel(time=0, depth=0).thetao.plot(cmap="magma")
velocity = ds_fields.isel(time=0, depth=0).plot.quiver(
    x="longitude", y="latitude", u="uo", v="vo"
)
for traj in df.partition_by("particle_id", maintain_order=True):
    plt.plot(traj["x"], traj["y"], color="blue")
plt.ylim(-31, -30)
plt.xlim(31, 32.1)
plt.show()

## References

Gräwe, U. (2011). “Implementation of high-order particle-tracking schemes in a water column model.” _Ocean Modelling_, 36(1), 80–89. https://doi.org/10.1016/j.ocemod.2010.10.002

Gräwe, Deleersnijder, Shah & Heemink (2012). “Why the Euler scheme in particle tracking is not enough: The shallow-sea pycnocline test case.” _Ocean Dynamics_, 62(4), 501–514. https://doi.org/10.1007/s10236-012-0523-y

Maruyama, G. (1955). “Continuous Markov processes and stochastic equations.” _Rendiconti del Circolo Matematico di Palermo_, 4(1), 48.

van Sebille et al. (2018). “Lagrangian ocean analysis: Fundamentals and practices.” _Ocean Modelling_, 121, 49–75. https://doi.org/10.1016/j.ocemod.2017.11.008

Shah, S. H. A. M., Heemink, A. W., & Deleersnijder, E. (2011). “Assessing Lagrangian schemes for simulating diffusion on non-flat isopycnal surfaces.” _Ocean Modelling_, 39(3–4), 351–361. https://doi.org/10.1016/j.ocemod.2011.05.008
https://doi.org/10.1016/j.ocemod.2011.05.008

Shah, Primeau, Deleersnijder & Heemink (2017). “Tracing the Ventilation Pathways of the Deep North Pacific Ocean Using Lagrangian Particles and Eulerian Tracers.” _Journal of Physical Oceanography_, 47(6), 1261–1280. https://doi.org/10.1175/JPO-D-16-0098.1

Smagorinsky, J. (1963). “General circulation experiments with primitive equations. 1. The basic experiment.” _Monthly Weather Review_, 91, 99–164. https://doi.org/10.1175/1520-0493(1963)091%3C0099:GCEWTP%3E2.3.CO;2
